# 05 — Audio feature detection
This notebook inspects an audio file, visualizes waveform/MFCC/mel features, trains the local acoustic baseline, and predicts only when a trained artifact exists. Add licensed/consented samples using `data/README.md`. No metrics or audio examples are bundled.

In [ ]:
from pathlib import Path
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from src.audio_detector import extract_audio_features
DATA_DIR = Path('data/audio')
candidates = list(DATA_DIR.glob('real/*')) if DATA_DIR.exists() else []
audio_path = next((p for p in candidates if p.is_file()), None)
print('Example:', audio_path if audio_path else 'Add a real sample under data/audio/real/')

In [ ]:
if audio_path:
    signal, sr = librosa.load(audio_path, sr=16000, mono=True, duration=60)
    print(f'Duration analyzed: {len(signal)/sr:.2f} sec; sample rate: {sr} Hz')
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    librosa.display.waveshow(signal, sr=sr, ax=axes[0]); axes[0].set_title('Waveform')
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=20)
    librosa.display.specshow(mfcc, x_axis='time', ax=axes[1]); axes[1].set_title('MFCC coefficients')
    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=64, power=2)
    librosa.display.specshow(librosa.power_to_db(mel, ref=np.max), sr=sr, x_axis='time', y_axis='mel', ax=axes[2]); axes[2].set_title('Mel spectrogram (dB)')
    plt.tight_layout(); plt.show()
    features, metadata = extract_audio_features(audio_path)
    print('Feature vector:', features.shape, 'metadata:', metadata)

MFCC summarize the short-time spectral envelope; the mel spectrogram distributes energy on a perceptual frequency scale. Spectral centroid, bandwidth and rolloff describe spectral shape; zero-crossing rate describes sign changes; RMS summarizes frame energy. This system aggregates means and standard deviations over time into 178 features. These are generic acoustic descriptors, not proof of a synthetic voice.

In [ ]:
# Runs the same stratified training/evaluation script as the CLI and writes real metrics/artifact.
if DATA_DIR.exists():
    from src.train_audio import main
    import sys
    _old_argv = sys.argv
    sys.argv = ['train_audio.py', '--data-dir', str(DATA_DIR)]
    try:
        main()
    finally:
        sys.argv = _old_argv

In [ ]:
from src.audio_detector import predict_audio
import json
metrics_file = Path('models/audio_metrics.json')
if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text())
    print('Held-out metrics:', json.dumps(metrics, indent=2))
    import seaborn as sns
    cm = metrics['confusion_matrix_labels_real_synthetic']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['real', 'synthetic'],
                yticklabels=['real', 'synthetic'])
    plt.title('Held-out test confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()
if audio_path and Path('models/audio_model.pkl').is_file():
    prediction = predict_audio(audio_path)
    print('Prediction:', prediction)
else:
    print('Prediction requires an input recording and a trained models/audio_model.pkl artifact.')